# Advanced use cases

*Each section stands on its own and runs against the same in-memory fakes as the story. These are the parts of the
SDK that go beyond what the Space API does today; where a section needs something the API does not have, it says so
and shows the shape we would propose.*

In [1]:
import syfthub
from syfthub import AsyncHub, Options
from syfthub.testing import World, MockTransport

world = World()                                     # the mock's stand-in for the outside world
hub = AsyncHub(options=Options(http_client=MockTransport(world)))
await hub.login(username="alice", password="secret")

Hub wallet,"$1.50 (MPP only, experimental)"


## 1. Filters that travel to the Space

A filter on document metadata has two possible homes. If the Space applies it, the filter runs *before* `limit`: you
get `limit` matching documents and, on a per-document price, pay only for those. If the SDK applies it, the filter runs
*after* `limit` on whatever came back: you may get fewer than `limit`, and you have paid for the ones it hid.

Today a Space does not filter. The proposal: a Space publishes the metadata fields it can filter on along with its
endpoint (a `filterable` list on the Hub record), accepts a `filters` object in the query body for those fields, and
echoes what it applied as `filters_applied`. The SDK then has one verb, `filter(...)`, and tells you per source which
home each key gets. The endpoint detail shows what a Space advertises:

In [2]:
await hub.endpoints.get("carol/papers")

Endpoint(carol/papers, data_source, free)

In [3]:
await hub.endpoints.get("dave/notes")      # advertises nothing: filters on Dave's notes stay on the client

Endpoint(dave/notes, data_source, $0.02/document via xendit)

Same filter, three sources, three documents each. Carol's papers and Erin's registry take `published` at the Space;
Dave's notes do not, and they are priced per document, so the pre-flight warns that the hidden ones are still billed.

In [4]:
plan = hub.search("adverse events in phase 3 trials", sources=["carol/papers", "erin/trials", "dave/notes"], limit=3)
plan = plan.filter(published_gte="2024-06-01")
await plan.preflight()

SearchPlan "adverse events in phase 3 trials"  est. max $0.05  (2 of 3 will be sent)
  carol/papers             free                             max $0.00    ready
  erin/trials              $0.05/request via stripe         max $0.05    ready
  dave/notes               $0.02/document via xendit        max $0.06    needs credits  (held)

In [5]:
invoice = await plan.top_up("dave/notes", "starter")
world.pay(invoice)                                        # mock: paid at the link
results = await plan.execute()
results

endpoint,returns,outcome,result,cost,latency
carol/papers,references,returned,3 documents · filtered at the Space,-,587 ms
erin/trials,references,returned,1 document · filtered at the Space,$0.05,381 ms
dave/notes,references,returned,2 documents of 3 returned,$0.06,354 ms
source,policy,status,amount,wallet / rail,transaction
erin/trials,stripe_per_request,charged,$0.05,w-erin-stripe-usd,ledger-894338
dave/notes,xendit_per_document,charged,$0.06,w-dave-xendit-usd,ledger-711114


Carol returned three documents, all from mid-2024 or later: the Space filtered first and then applied the limit.
Without the Space's help she would have got two, because her third-best match is from 2023. Dave's Space did exactly
that: it returned its three best matches, the SDK hid the 2023 one, and the charge is still for three.

In [6]:
results["carol/papers"]

SourceResult(carol/papers, returned, 3 docs)

In [7]:
results["dave/notes"]

policy_type,status,amount,transaction / reason_code,
xendit_per_document,charged,$0.06,ledger-711114,


What actually went over the wire. The request body carries the proposed `filters` object only for Spaces that
advertise the field; the response echoes what the Space applied.

In [8]:
results["carol/papers"].request["filters"], results["carol/papers"].raw["filters_applied"]

({'published': {'gte': '2024-06-01'}}, {'published': {'gte': '2024-06-01'}})

In [9]:
"filters" in results["dave/notes"].request        # nothing was sent: the SDK filtered after the fact

False

A filter can split. Erin advertises only `published`, so `author` stays on the client for her and goes to the Space
for Carol. The pre-flight shows the split per source before anything is spent.

In [10]:
await hub.search("adverse events in phase 3 trials", sources=["carol/papers", "erin/trials"], limit=3) \
         .filter(published_gte="2024-01-01", author="R. Chitrakoot").preflight()

endpoint,pricing,max cost,your balance,pre-flight,filters,
carol/papers,free,-,-,ready,"at the Space published_gte, author",
erin/trials,$0.05/request via stripe,$0.05,$0.95,ready,"at the Space published_gtehere, after limit author",deducted from wallet erin stripe USD


The same `filter` verb works on a chat that searches every turn: the filter goes with each turn's search, to the
Space where it can.

In [11]:
live = hub.chat("bob/llama-3", sources=["carol/papers", "erin/trials"]).filter(published_gte="2024-06-01")
await live.send("What were the serious adverse event rates?")

[1],carol/papers
[2],carol/papers
[4],erin/trials


In [12]:
live.last.results         # that turn's search, filtered at both Spaces

endpoint,returns,outcome,result,cost,latency
carol/papers,references,returned,3 documents · filtered at the Space,-,315 ms
erin/trials,references,returned,1 document · filtered at the Space,$0.05,369 ms
source,policy,status,amount,wallet / rail,transaction
erin/trials,stripe_per_request,charged,$0.05,w-erin-stripe-usd,ledger-949949


## 2. A room of models from the start

`results.chat` takes one model or several. Each gets its own pre-flight row; one that is short of credits is held
and answers with a top-up card, the others answer normally. Nothing blocks on the one that cannot pay.

In [13]:
results = await hub.search("adverse events in phase 3 trials", sources=["carol/papers", "olga/trials-assistant"]).execute()
room = results.chat(["ivan/gpt-mini", "bob/llama-3"])
await room.preflight()

Chat(models=['ivan/gpt-mini', 'bob/llama-3'], via=direct, turns=0)

In [14]:
replies = await room.send("Summarise the serious adverse events.")
replies

wallet,ivan · stripe USD w-ivan-stripe-usd
your balance,$0.00 · needed $0.02
bundles,starter: $5.00 pro: $20.00
[1],carol/papers
[2],carol/papers
[3],carol/papers
[7],olga/trials-assistant


In [15]:
[(r.model.path, r.outcome.value, r.reason.value if r.reason else None) for r in replies]

[('ivan/gpt-mini', 'held', 'no_credits'), ('bob/llama-3', 'returned', None)]

Credits fix the held tab. The top-up names the model when more than one could be short; here only Ivan is, so the
name is optional. Sending again asks both, and both answer.

In [16]:
invoice = await room.top_up("ivan/gpt-mini", bundle="starter")
world.pay(invoice)                                        # mock: paid at the link
replies = await room.send("Summarise the serious adverse events.")
replies

[1],carol/papers
[2],carol/papers
[3],carol/papers
[7],olga/trials-assistant
[1],carol/papers
[2],carol/papers
[3],carol/papers
[7],olga/trials-assistant


Who answers can change between turns. `room.add` brings a model in from the next question on, briefed with the
lead tab's transcript by default; `room.remove` stops asking one. Each model sees the shared questions and only
its own earlier answers.

In [17]:
await room.add("lena/sahabat-ai")      # billed in rupiah, from Lena's IDR wallet
await room.send("Were any of them fatal?")

[5],carol/papers
[5],carol/papers
[5],carol/papers


In [18]:
await room.remove("ivan/gpt-mini")
await room.send("And in one line?")

[5],carol/papers
[5],carol/papers


`room.spent` is the room's total across every turn and every model, kept per currency because models sit behind
different wallets: Ivan bills in dollars, Lena in rupiah, and the SDK never converts. `room.spent_by(path)` is one
tab's share. The card shows the same numbers: the totals in the header, the share on each tab.

In [19]:
room.spent, {m.path: room.spent_by(m.path) for m in room.models}

({'USD': Money(0.04, 'USD'), 'IDR': Money(600.0, 'IDR')},
 {'ivan/gpt-mini': {'USD': Money(0.04, 'USD')},
  'bob/llama-3': {},
  'lena/sahabat-ai': {'IDR': Money(600.0, 'IDR')}})

In [20]:
room

Chat(models=['bob/llama-3', 'lena/sahabat-ai'], via=direct, turns=4)

**What this needs from the platform.** Three small additions, none of which change the response shape for clients
that do not filter: `filterable: [str]` on the published endpoint and the Hub's public record; `filters` in the
query body, `{field: {op: value}}` with ops `eq | gte | gt | lte | lt | contains`; `filters_applied` echoed in the
response. Until then the SDK behaves as if every Space advertised nothing, which is exactly the client-side path.